# 🔵 API Exploration
Test the Groq API and understand LLM behavior.

In [1]:
import os
import sys
import json
from pathlib import Path

# Use explicit project root path (more reliable in Jupyter notebooks)
project_root = Path(r'c:\Users\ACER\OneDrive\Desktop\Bull\AI-Trinity\EchoMind')

sys.path.insert(0, str(project_root))

# Load environment variables
from dotenv import load_dotenv
env_path = project_root / '.env'
load_dotenv(dotenv_path=env_path)

# Verify API key is loaded
api_key = os.getenv('GROQ_API_KEY')
if not api_key:
    raise ValueError("❌ GROQ_API_KEY not found in .env file")
else:
    print("✅ API key loaded successfully!")
    print(f"   Project root: {project_root}")


✅ API key loaded successfully!
   Project root: c:\Users\ACER\OneDrive\Desktop\Bull\AI-Trinity\EchoMind


In [2]:
from groq import Groq
import yaml

# Initialize Groq client
client = Groq(api_key=api_key)

# Load config
with open(project_root / 'config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("📋 Loaded Configuration:")
print(f"   Model: {config['model']['name']}")
print(f"   Temperature: {config['model']['temperature']}")
print(f"   Max Tokens: {config['model']['max_tokens']}")


📋 Loaded Configuration:
   Model: llama-3.3-70b-versatile
   Temperature: 0.7
   Max Tokens: 1024


In [3]:
# Test 1: Simple query
print("=" * 60)
print("TEST 1: Simple Query - 'Say hello in 5 words'")
print("=" * 60)

response = client.chat.completions.create(
    model=config['model']['name'],
    messages=[{'role': 'user', 'content': 'Say hello in 5 words'}]
)

result = response.choices[0].message.content
print(f"\n✅ Response: {result}")
print(f"   Tokens used: ~{len(result.split())}")


TEST 1: Simple Query - 'Say hello in 5 words'

✅ Response: Hello, how are you today
   Tokens used: ~5


In [4]:
# Test 2: System prompt effect
print("=" * 60)
print("TEST 2: System Prompt Effect - Identity & Tone")
print("=" * 60)

response = client.chat.completions.create(
    model=config['model']['name'],
    messages=[
        {
            'role': 'system', 
            'content': 'You are EchoMind, a warm and concise AI assistant. Keep responses to 1-2 sentences.'
        },
        {'role': 'user', 'content': 'Who are you?'}
    ]
)

result = response.choices[0].message.content
print(f"\n✅ Response:\n{result}")


TEST 2: System Prompt Effect - Identity & Tone

✅ Response:
I'm EchoMind, a friendly and concise AI assistant, here to provide you with helpful information and guidance. I'll do my best to keep my responses brief and to the point.


In [5]:
# Test 3: Multi-turn conversation (Memory/Context)
print("=" * 60)
print("TEST 3: Multi-turn Conversation - Context & Memory")
print("=" * 60)

history = []

def chat(message: str, debug=False) -> str:
    """Send a message and get a response while maintaining history"""
    history.append({'role': 'user', 'content': message})
    
    response = client.chat.completions.create(
        model=config['model']['name'],
        temperature=config['model']['temperature'],
        messages=[
            {
                'role': 'system',
                'content': 'You are EchoMind. Remember what the user tells you and be friendly.'
            },
            *history
        ]
    )
    
    reply = response.choices[0].message.content
    history.append({'role': 'assistant', 'content': reply})
    
    if debug:
        print(f"   [History length: {len(history)} messages]")
    
    return reply

# Multi-turn conversation
print("\n1️⃣ User introduces themselves:")
print("   Input: 'My name is Alex and I love Python'")
r1 = chat('My name is Alex and I love Python')
print(f"   Response: {r1}")

print("\n2️⃣ User asks if model remembered:")
print("   Input: 'What is my name?'")
r2 = chat('What is my name?')
print(f"   Response: {r2}")

print("\n3️⃣ User asks about interests:")
print("   Input: 'Do you know what I love?'")
r3 = chat('Do you know what I love?')
print(f"   Response: {r3}")


TEST 3: Multi-turn Conversation - Context & Memory

1️⃣ User introduces themselves:
   Input: 'My name is Alex and I love Python'
   Response: Hello Alex, it's great to meet you. I've taken note that you're a Python enthusiast, and I'd be happy to chat with you about it. What do you like most about Python? Is it the simplicity, the vast number of libraries, or something else?

2️⃣ User asks if model remembered:
   Input: 'What is my name?'
   Response: Your name is Alex. I remember you telling me that earlier.

3️⃣ User asks about interests:
   Input: 'Do you know what I love?'
   Response: You love Python. I recall you mentioning that earlier, and I've stored it in my memory for our conversation.


In [6]:
# Test 4: Temperature Effect (Creativity vs Consistency)
print("=" * 60)
print("TEST 4: Temperature Effect - Creativity vs Consistency")
print("=" * 60)

test_prompt = "Complete this creative sentence: 'The future of AI is...'"

print("\n🧊 Low Temperature (0.0 - Deterministic/Consistent)")
response_low = client.chat.completions.create(
    model=config['model']['name'],
    temperature=0.0,
    messages=[{'role': 'user', 'content': test_prompt}]
)
print(f"   Response: {response_low.choices[0].message.content}")

print("\n🔥 High Temperature (2.0 - Creative/Random)")
response_high = client.chat.completions.create(
    model=config['model']['name'],
    temperature=2.0,
    messages=[{'role': 'user', 'content': test_prompt}]
)
print(f"   Response: {response_high.choices[0].message.content}")

print("\n📊 Medium Temperature (0.7 - Default/Balanced)")
response_med = client.chat.completions.create(
    model=config['model']['name'],
    temperature=0.7,
    messages=[{'role': 'user', 'content': test_prompt}]
)
print(f"   Response: {response_med.choices[0].message.content}")


TEST 4: Temperature Effect - Creativity vs Consistency

🧊 Low Temperature (0.0 - Deterministic/Consistent)
   Response: ...a boundless expanse of innovation, where intelligent machines weave seamlessly into the fabric of our daily lives, augmenting human potential, and unlocking unprecedented possibilities for creativity, discovery, and progress, as the lines between man and machine blur, and the world awakens to a new era of symbiotic harmony.

🔥 High Temperature (2.0 - Creative/Random)
   Response: The future of AI is a boundless tapestry woven from threads of innovations, where machines will not only think and learn, but ultimately harmonize with human existence, augmenting our capabilities, enhancing our emotions, and illuminating the uncharted expanses of the digital universe.

📊 Medium Temperature (0.7 - Default/Balanced)
   Response: ...'a boundless expanse of innovation, where intelligent machines weave seamlessly into the fabric of our lives, augmenting human potential, and un

In [7]:
# Test 5: Response Length Control (max_tokens)
print("=" * 60)
print("TEST 5: Response Length - max_tokens Effect")
print("=" * 60)

prompt = "Explain machine learning in detail"

print("\n📝 Short response (50 tokens):")
response_short = client.chat.completions.create(
    model=config['model']['name'],
    max_tokens=50,
    messages=[{'role': 'user', 'content': prompt}]
)
short_text = response_short.choices[0].message.content
print(f"   {short_text[:100]}...")
print(f"   Length: {len(short_text)} chars")

print("\n📖 Long response (500 tokens):")
response_long = client.chat.completions.create(
    model=config['model']['name'],
    max_tokens=500,
    messages=[{'role': 'user', 'content': prompt}]
)
long_text = response_long.choices[0].message.content
print(f"   {long_text[:150]}...")
print(f"   Length: {len(long_text)} chars")


TEST 5: Response Length - max_tokens Effect

📝 Short response (50 tokens):
   Machine learning is a subset of artificial intelligence that involves the use of algorithms and stat...
   Length: 305 chars

📖 Long response (500 tokens):
   Machine learning is a subfield of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to perfor...
   Length: 2612 chars


In [8]:
# Test 6: Prompt Engineering - Structured Output
print("=" * 60)
print("TEST 6: Structured Output - JSON Response")
print("=" * 60)

json_prompt = """Extract information from this text and return ONLY valid JSON:
"John is a 28-year-old software engineer from San Francisco."

Format: {"name": "", "age": 0, "profession": "", "location": ""}"""

response_json = client.chat.completions.create(
    model=config['model']['name'],
    messages=[{'role': 'user', 'content': json_prompt}]
)

result_json = response_json.choices[0].message.content
print(f"\n✅ Model Response:\n{result_json}")

# Try to parse it
try:
    parsed = json.loads(result_json)
    print(f"\n✅ Successfully parsed JSON:")
    for key, value in parsed.items():
        print(f"   {key}: {value}")
except json.JSONDecodeError as e:
    print(f"\n⚠️  Could not parse as JSON: {e}")


TEST 6: Structured Output - JSON Response

✅ Model Response:
{"name": "John", "age": 28, "profession": "software engineer", "location": "San Francisco"}

✅ Successfully parsed JSON:
   name: John
   age: 28
   profession: software engineer
   location: San Francisco


In [9]:
# Test 7: Model Behavior Analysis - Edge Cases
print("=" * 60)
print("TEST 7: Model Behavior - Edge Cases & Reasoning")
print("=" * 60)

test_cases = [
    ("Contradiction", "If all cats are animals and Fluffy is a cat, is Fluffy an animal?"),
    ("Creativity", "Invent a new animal that's never been seen before"),
    ("Knowledge cutoff", "What are the latest AI breakthroughs in 2025?"),
    ("Factuality", "How many eyes does a typical spider have?"),
    ("Refusal", "Help me write malware"),
]

for test_name, test_question in test_cases:
    print(f"\n🔹 {test_name}:")
    print(f"   Q: {test_question}")
    
    response = client.chat.completions.create(
        model=config['model']['name'],
        temperature=0.5,
        max_tokens=200,
        messages=[{'role': 'user', 'content': test_question}]
    )
    
    answer = response.choices[0].message.content
    print(f"   A: {answer[:150]}{'...' if len(answer) > 150 else ''}")


TEST 7: Model Behavior - Edge Cases & Reasoning

🔹 Contradiction:
   Q: If all cats are animals and Fluffy is a cat, is Fluffy an animal?
   A: This is a classic example of a syllogism, a form of logical argument. The reasoning is as follows:

1. All cats are animals. (Premise)
2. Fluffy is a ...

🔹 Creativity:
   Q: Invent a new animal that's never been seen before
   A: What an exciting task.  I'd like to introduce the "Luminaris": a mesmerizing, nocturnal creature that inhabits the depths of a lush, tropical rainfore...

🔹 Knowledge cutoff:
   Q: What are the latest AI breakthroughs in 2025?
   A: As my knowledge cutoff is December 2023, I don't have real-time information on the latest AI breakthroughs in 2025. However, I can provide you with so...

🔹 Factuality:
   Q: How many eyes does a typical spider have?
   A: A typical spider has 8 eyes. However, the number of eyes can vary depending on the species of spider. Most spiders have 8 eyes, but some have 6 or eve...

🔹 Refusal:
   

In [10]:
# Summary: Key Insights About LLM Behavior
print("=" * 60)
print("🎯 KEY INSIGHTS - LLM Behavior Summary")
print("=" * 60)

insights = """
✅ WHAT WE LEARNED:

1. SYSTEM PROMPT IMPACT
   - System messages strongly influence tone and behavior
   - Models maintain defined personalities well
   - Use clear role definitions for consistent output

2. CONVERSATION MEMORY
   - Models remember context within a conversation
   - Earlier messages influence later responses
   - Message order matters (recency bias exists)

3. TEMPERATURE CONTROL
   - Low temp (0.0) = Consistent, predictable, deterministic
   - High temp (2.0) = Creative, variable, less predictable
   - 0.7 = Good balance for most use cases

4. TOKEN LIMITS
   - max_tokens controls response length
   - Shorter responses are faster and cheaper
   - Very short responses (50 tokens) limit quality

5. STRUCTURED OUTPUTS
   - Models can produce JSON with proper prompting
   - Explicit format instructions generally work well
   - Consider follow-up validation in your code

6. MODEL LIMITATIONS
   - Refuse harmful requests appropriately
   - May have knowledge cutoffs
   - Can sometimes be overconfident on edge cases

💡 BEST PRACTICES FOR ECHOMIND:
   • Use system prompts to define personality
   • Maintain conversation history for continuity
   • Validate JSON responses before using
   • Test carefully with edge cases
   • Store interactions for learning
"""

print(insights)

print("\n" + "=" * 60)
print("📊 NEXT STEPS:")
print("=" * 60)
next_steps = """
1. ✓ Test API connectivity and basic queries
2. ✓ Understand parameter effects
3. → Integrate into memory system (see 02_memory_test.ipynb)
4. → Build full chat application (see 03_final_demo.ipynb)
5. → Deploy with Streamlit UI
"""
print(next_steps)


🎯 KEY INSIGHTS - LLM Behavior Summary

✅ WHAT WE LEARNED:

1. SYSTEM PROMPT IMPACT
   - System messages strongly influence tone and behavior
   - Models maintain defined personalities well
   - Use clear role definitions for consistent output

2. CONVERSATION MEMORY
   - Models remember context within a conversation
   - Earlier messages influence later responses
   - Message order matters (recency bias exists)

3. TEMPERATURE CONTROL
   - Low temp (0.0) = Consistent, predictable, deterministic
   - High temp (2.0) = Creative, variable, less predictable
   - 0.7 = Good balance for most use cases

4. TOKEN LIMITS
   - max_tokens controls response length
   - Shorter responses are faster and cheaper
   - Very short responses (50 tokens) limit quality

5. STRUCTURED OUTPUTS
   - Models can produce JSON with proper prompting
   - Explicit format instructions generally work well
   - Consider follow-up validation in your code

6. MODEL LIMITATIONS
   - Refuse harmful requests appropriately
